**偶发信号**和和**多探针覆盖范围下的智能设备信号**是需要清洗的内容，
- 偶发信号的特征为数据量极少且到达探针的数量少
- 多探针覆盖范围下的智能设备信号特征为数据量多且到达探针的数量为1-4个


另外主要还有**常驻人群**，**游客**，**通勤人群**三类数据，但由于信号干扰等原因，不同数据之间的分隔基准并不明确，同时人群种类可能并不仅仅只有前文所列的三类，不同类型也有可能因为信号干扰、行为模式不同而导致数据差异，因此通过简单的逻辑判断清洗效果和分类效果都不甚理想。

尝试通过kmeans或dbscan聚类来进行分类

## DBSCAN聚类
DBSCAN（Density-Based Spatial Clustering of Applications with Noise）是一种基于密度的聚类算法，可以将数据点分成不同的簇，并且能够识别噪声点（不属于任何簇的点）。

DBSCAN聚类算法的基本思想是：

在给定的数据集中，根据每个数据点周围其他数据点的密度情况，将数据点分为核心点、边界点和噪声点。

- 核心点是周围某个半径内有足够多其他数据点的数据点；
- 边界点是不满足核心点要求，但在某个核心点的半径内的数据点；
- 噪声点则是不满足任何条件的点。


## 轮廓系数、卡林斯基-哈拉巴斯指数、DBCV进行评估

- **轮廓系数（silhouette_score）**，轮廓系数指标不关注样本的实际类别，而是通过分析聚类结果中样本的内聚度和分离度两种因素来给出成绩，取值范围为（-1，1），**值越大代表聚类的结果越合理**。
- **卡林斯基-哈拉巴斯指数(Calinski-Harabasz Index)**，组内离散程度低，**协方差矩阵**的迹就会越小，同时，组间离散程度大，**协方差矩阵**的迹就会越大。因此calinski_harabasz**指数越高越好**。
  
 $$ s(k)=\frac{T_r(B_k)}{T_r(W_k)}*\frac{N-k}{k-1} $$

其中$N$是数据集的样本量，$k$为簇的个数，$B_k$是组间离散矩阵,即不同簇之间的协方差矩阵。$W_k$是簇内离散矩阵，即一个簇内数据的协方差矩阵，而$Tr$表示矩阵的迹。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mat
import matplotlib.cm as cm
import os
import datetime
import time
from tqdm import tqdm
import plotly.express as px
import plotly.io as pio

import sys
sys.path.append('../plot/')
import myplot

sys.path.append('../utils/')
import utils
import DBSCAN
from DBSCAN import My_DBSCAN,My_DBSCAN_MATRIX

# 设置plotly默认主题
pio.templates.default = 'plotly_white'

mat.rcParams['font.family'] = 'SimHei'
mat.rcParams['font.sans-serif'] = 'SimHei'

import warnings
warnings.filterwarnings('ignore')

from collections import namedtuple

from importlib import reload

In [2]:
df_read = pd.read_csv(os.getcwd()+'/dacang/cleaned_data/dacang_cleaned_data_advanced.csv')
df_read

,M,Dur,A_count,Count,oriMac,date,Moment
0,"7.5-48,74,38,155,214,98",21.83,6.0,147.0,"48,74,38,155,214,98",7.5,22.73
1,"7.5-20,178,229,137,47,142",17.02,4.0,38.0,"20,178,229,137,47,142",7.5,20.33
2,"7.6-4,207,140,11,128,186",23.98,4.0,947.0,"4,207,140,11,128,186",7.6,12.00
3,"7.6-208,151,254,40,222,169",23.97,4.0,910.0,"208,151,254,40,222,169",7.6,11.98
4,"7.5-48,74,38,133,29,213",17.00,5.0,49.0,"48,74,38,133,29,213",7.5,20.35
...,...,...,...,...,...,...,...
1063,"8.7-176,70,146,156,89,69",9.38,5.0,179.0,"176,70,146,156,89,69",8.7,17.18
1064,"8.7-44,220,173,148,169,213",7.48,4.0,171.0,"44,220,173,148,169,213",8.7,17.82
1065,"8.7-124,161,119,201,50,40",9.78,6.0,260.0,"124,161,119,201,50,40",8.7,18.98
1066,"8.7-152,207,83,118,160,172",19.45,4.0,54.0,"152,207,83,118,160,172",8.7,12.45


In [3]:
df_cluster = utils.Normalize_df(df_read[['Dur','Count','Moment']])
df_cluster

,Dur,Count,Moment
0,9.065624,0.578723,9.575552
1,6.975228,0.114894,8.556876
2,10.000000,3.982979,5.021222
3,9.995654,3.825532,5.012733
4,6.966536,0.161702,8.565365
...,...,...,...
1063,3.654933,0.714894,7.219864
1064,2.829205,0.680851,7.491511
1065,3.828770,1.059574,7.983871
1066,8.031291,0.182979,5.212224


# DBSCAN

In [5]:
reload(DBSCAN)
#多次聚类，确定效果最佳的eps与min_samples
results = My_DBSCAN_MATRIX(df_cluster,
                 init_eps = 0.1,
                 step_eps = 0.02,
                 epoch_eps = 150,
                 init_mSamples = 5,
                 step_mSamples = 1,
                 epoch_mSamples = 10,
                 cal_DBCV = True
                 )

on dbscan...: 100%|██████████| 150/150 [15:45:29<00:00, 378.19s/it]   


In [6]:
#保存聚类数据
df_dbscan_result = pd.DataFrame({
    "cluster_num":list(map(lambda x : x.cluster_num, results)),
    "noise_num":list(map(lambda x : x.noise_num, results)),
    "dbcv":list(map(lambda x : x.dbcv, results)),
    "silhouette":list(map(lambda x : x.silhouette, results)),
    "calinski":list(map(lambda x : x.calinski, results)),
    "eps":list(map(lambda x : x.eps, results)),
    "min_samples":list(map(lambda x : x.min_samples, results)),
    })

df_dbscan_result.to_csv(os.getcwd()+"/dacang/cluster/dbscan_result_eliNoise_1201.csv",index=False)
#np.save(os.getcwd()+"/dacang/cluster/cluster_mat_result_1128.npy",cluster_labels)

### 聚类结果评估

In [56]:
df_dbscan_result = pd.read_csv(os.getcwd()+"/dacang/cluster/dbscan_result_eliNoise_1201.csv")
df_dbscan_result = df_dbscan_result[df_dbscan_result.eps<2]
utils.ShowClusterResult(df_dbscan_result,['cluster_num','noise_num','dbcv','silhouette','calinski'])

In [61]:
#cut data by noise_num
df_cut = utils._cut_Data_By_Thre(df_dbscan_result,['cluster_num','noise_num','dbcv','silhouette','calinski'],
                                           450,'noise_num','>')
#cut data by cluster_num
df_cut = utils._cut_Data_By_Thre(df_cut,['cluster_num','noise_num','dbcv','silhouette','calinski'],
                                           15,'cluster_num','>')

In [62]:
utils.ShowClusterResult(df_cut,['cluster_num','noise_num','dbcv','silhouette','calinski'],
                        cut_thre = 3,cut_col_name = 'cluster_num',
                        cut_mode = '<')

## 确定eps和min_samples后分析

In [73]:
#eps = 0.64 , min_samples = 11
df_cluster = utils.Normalize_df(df_read[['Dur','Count','Moment']])
result = My_DBSCAN(df_cluster,0.64,11)
df_cluster['label'] = result.labels
df_cluster['A_count'] = df_read.A_count
print('noise_num:',df_cluster.label.value_counts()[-1])
print('cluster_num:',len(set(df_cluster.label)))
myplot.Scatter_3D(df_cluster,'Dur','Count','Moment',species_name='label',color_name='label')


noise_num: 227
cluster_num: 7
